## Tarea 2
Programa Experto en Inteligencia Artificial con Python  
ML2006 - Visualización e Interpretación de Datos  
Estudiante: Natalia Bonilla Villalobos.  
**GeoPandas**

<a id="menu"></a>

# Menú
- [Ejercicio 1](#ej1)
- [Ejercicio 2](#ej2)
- [Ejercicio 3](#ej3)
- [Ejercicio 4](#ej4)

In [1]:
import numpy as np
import pandas as pd
import geopandas as gp
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import folium
from fuzzy_pandas import fuzzy_merge
from io import BytesIO
import base64

Para los siguientes ejercicios utilice la base de datos `Football.sqlite`, de la cual se obtuvo el dataframe que contiene las tablas de posiciones por temporada, pero filtre únicamente para la liga española: `Spain LIGA BBVA`, (lo llamaremos dataframe **A**).  
También utilice el archivo `LaLiga_Stadiums_2010_2025.csv`, el cual contiene información de la ubicación geográfica de estadios en España, así como el nombre del Club y ciudad (lo llamaremos dataframe **B**).

<a id="ej1"></a>
# Ejercicio 1 

[25 puntos] Se requiere mostrar información obtenida de A sobre un mapa, utilizando las coordenadas de los estadios del dataframe **B**; no obstante, el nombre de los equipos (club) en el dataframe **B** no es exactamente igual al nombre de los equipos que se encuentran en la bases de datos (**A**).  

Previo a la visualización de la información sobre el mapa:
- Cargue el archivo con la información de los estadios (dataframe **B**).
- Obtenga un listado con los distintos equipos (nombres) presentes en A, esto únicamente para la liga española (`Spain LIGA BBVA`), (lo llamaremos dataframe **C**).
- Utilizando la función `fuzzy_merge` de la biblioteca fuzzy_pandas, u otro mecanismo, combine el dataframe de estadios (**B**) con el dataframe de los nombres únicos de los equipos
(**C**). Cuando utilice `fuzzy_merge`, puede especificar el método `method=‘levenshtein’` para comparar los textos, y un umbral máximo `threshold=0.6`.

- Dependiendo del método para combinar los dataframes, el dataframe resultante podría tener registros duplicados, es decir, varios estadios relacionados con varios equipos. Dicho
esto, desarrolle una lógica para obtener un registro por equipo y estadio; este nuevo dataframe sin duplicados es nuestro nuevo dataframe **D**.
- Con la información ahora de los equipos y los estadios (D), combine ahora esos datos con el dataframe inicial (**A**) de la tabla de posiciones por temporada de la liga española (**A**), llamaremos a este dataframe final **F**.
- Seleccionando solo una de las temporadas (definida a su discreción), muestre el dataframe resultante **F**.


[↑ Volver al Menú](#menu)


### Parte 1

- Cargue el archivo con la información de los estadios (dataframe **B**).

In [2]:
df_b = pd.read_csv("./../LaLiga_Stadiums_2010_2025.csv", encoding="cp1252") # or ISO-8859-1
df_b

,Club,Stadium,City,Latitude,Longitude
0,Alavés,Mendizorrotza,Vitoria-Gasteiz,42.84610,-2.67270
1,Almería,Estadio de los Juegos Mediterráneos,Almería,36.84000,-2.46700
2,Athletic Bilbao,San Mamés,Bilbao,43.26420,-2.94940
3,Atlético Madrid,Metropolitano,Madrid,40.43620,-3.59950
4,Barcelona,Camp Nou,Barcelona,41.38090,2.12280
5,Barcelona (2023-2024),Estadi Olímpic Lluís Companys,Barcelona,41.36470,2.15560
6,Betis,Benito Villamarín,Seville,37.35640,-5.98160
7,Celta Vigo,Balaídos,Vigo,42.21180,-8.73940
8,Deportivo La Coruña,Riazor,A Coruña,43.36860,-8.41890
9,Eibar,Ipurua,Eibar,43.18330,-2.46670


### Parte 2
- Obtenga un listado con los distintos equipos (nombres) presentes en A, esto únicamente para la liga española (`Spain LIGA BBVA`), (lo llamaremos dataframe **C**).


In [3]:
conn = sqlite3.connect('../../w1/Football.sqlite')
df_a = pd.read_sql_query("""
select
    Liga, Temporada, Equipo, Equipo_longName,
    sum(Total_Puntos) as Total_Puntos,
    sum(Partidos_Jugados) as Partidos_Jugados,
    sum(Partidos_Empatados) as Partidos_Empatados,
    sum(Partidos_Perdidos) as Partidos_Perdidos,
    sum(GF) as GF,
    sum(GC) as GC,
    sum(GF) - sum(GC) as Diferencia_Goles
from(
    -- Local
    select l.name as Liga,
           m.season as Temporada,
           t.team_short_name as Equipo,
           t.team_long_name as Equipo_longName,
           case
               when m.home_team_goal > m.away_team_goal then 3
               when m.home_team_goal = m.away_team_goal then 1
               else 0
           end as Total_Puntos,
           1 as Partidos_Jugados,
           case when m.home_team_goal = m.away_team_goal then 1 else 0 end as Partidos_Empatados,
           case when m.home_team_goal < m.away_team_goal then 1 else 0 end as Partidos_Perdidos,
           m.home_team_goal as GF,
           m.away_team_goal as GC
    from Match m
    inner join (select distinct team_api_id, team_short_name, team_long_name from Team) t
        on m.home_team_api_id = t.team_api_id
    inner join League l
        on m.league_id = l.id

    UNION ALL

    -- Visitante
    select l.name as Liga,
           m.season as Temporada,
           t.team_short_name as Equipo,
           t.team_long_name as Equipo_longName,
           case
               when m.home_team_goal < m.away_team_goal then 3
               when m.home_team_goal = m.away_team_goal then 1
               else 0
           end as Total_Puntos,
           1 as Partidos_Jugados,
           case when m.home_team_goal = m.away_team_goal then 1 else 0 end as Partidos_Empatados,
           case when m.home_team_goal > m.away_team_goal then 1 else 0 end as Partidos_Perdidos,
           m.away_team_goal as GF,
           m.home_team_goal as GC
    from Match m
    inner join (select distinct team_api_id, team_short_name, team_long_name from Team) t
        on m.away_team_api_id = t.team_api_id
    inner join League l
        on m.league_id = l.id
) as tabla
group by
        Liga,
        Temporada,
        Equipo
""", conn)

conn.close()
df_a

,Liga,Temporada,Equipo,Equipo_longName,Total_Puntos,Partidos_Jugados,Partidos_Empatados,Partidos_Perdidos,GF,GC,Diferencia_Goles
0,Belgium Jupiler League,2008/2009,AND,RSC Anderlecht,77,34,5,5,75,30,45
1,Belgium Jupiler League,2008/2009,BAC,Beerschot AC,42,34,9,14,44,42,2
2,Belgium Jupiler League,2008/2009,CEB,KSV Cercle Brugge,47,34,5,15,48,53,-5
3,Belgium Jupiler League,2008/2009,CHA,Sporting Charleroi,43,34,7,15,43,48,-5
4,Belgium Jupiler League,2008/2009,CLB,Club Brugge KV,59,34,5,11,59,50,9
...,...,...,...,...,...,...,...,...,...,...,...
1452,Switzerland Super League,2015/2016,SIO,FC Sion,50,36,8,14,52,49,3
1453,Switzerland Super League,2015/2016,THU,FC Thun,41,36,11,15,45,54,-9
1454,Switzerland Super League,2015/2016,VAD,FC Vaduz,36,36,15,14,44,60,-16
1455,Switzerland Super League,2015/2016,YB,BSC Young Boys,69,36,9,7,78,47,31


In [4]:
spain_liga_bbva = df_a[
    (df_a['Liga'] == 'Spain LIGA BBVA')
]
spain_liga_bbva

,Liga,Temporada,Equipo,Equipo_longName,Total_Puntos,Partidos_Jugados,Partidos_Empatados,Partidos_Perdidos,GF,GC,Diferencia_Goles
1227,Spain LIGA BBVA,2008/2009,ALM,UD Almería,46,38,7,18,45,61,-16
1228,Spain LIGA BBVA,2008/2009,AMA,Atlético Madrid,67,38,7,11,80,57,23
1229,Spain LIGA BBVA,2008/2009,BAR,FC Barcelona,87,38,6,5,105,35,70
1230,Spain LIGA BBVA,2008/2009,BET,Real Betis Balompié,42,38,12,16,51,58,-7
1231,Spain LIGA BBVA,2008/2009,BIL,Athletic Club de Bilbao,44,38,8,18,47,62,-15
...,...,...,...,...,...,...,...,...,...,...,...
1372,Spain LIGA BBVA,2015/2016,SEV,Sevilla FC,52,38,10,14,51,50,1
1373,Spain LIGA BBVA,2015/2016,SOC,Real Sociedad,48,38,9,16,45,48,-3
1374,Spain LIGA BBVA,2015/2016,SPG,Real Sporting de Gijón,39,38,9,19,40,62,-22
1375,Spain LIGA BBVA,2015/2016,VAL,Valencia CF,44,38,11,16,46,48,-2


In [5]:
spain_liga_bbva['Equipo_longName'].unique()

array(['UD Almería', 'Atlético Madrid', 'FC Barcelona',
       'Real Betis Balompié', 'Athletic Club de Bilbao',
       'RC Deportivo de La Coruña', 'RCD Espanyol', 'Getafe CF',
       'RC Recreativo', 'RCD Mallorca', 'CD Numancia', 'CA Osasuna',
       'Real Madrid CF', 'Racing Santander', 'Sevilla FC',
       'Real Sporting de Gijón', 'Valencia CF', 'Villarreal CF',
       'CD Tenerife', 'Xerez Club Deportivo', 'Real Zaragoza',
       'Hércules Club de Fútbol', 'Levante UD', 'Málaga CF',
       'Real Sociedad', 'Granada CF', 'Rayo Vallecano',
       'RC Celta de Vigo', 'Elche CF', 'Real Valladolid', 'SD Eibar',
       'UD Las Palmas'], dtype=object)

In [6]:
df_c = spain_liga_bbva[['Equipo_longName']].drop_duplicates()

### Parte 3

- Utilizando la función `fuzzy_merge` de la biblioteca fuzzy_pandas, u otro mecanismo, combine el dataframe de estadios (**B**) con el dataframe de los nombres únicos de los equipos
(**C**). Cuando utilice `fuzzy_merge`, puede especificar el método `method=‘levenshtein’` para comparar los textos, y un umbral máximo `threshold=0.6`.

In [7]:
df_merge = fuzzy_merge(df_c, df_b,
                       left_on='Equipo_longName',
                       right_on='Club',
                       method='levenshtein',
                       threshold=0.6
)
df_merge

,Equipo_longName,Club,Stadium,City,Latitude,Longitude
0,UD Almería,Almería,Estadio de los Juegos Mediterráneos,Almería,36.84000,-2.46700
1,Atlético Madrid,Atlético Madrid,Metropolitano,Madrid,40.43620,-3.59950
2,FC Barcelona,Barcelona,Camp Nou,Barcelona,41.38090,2.12280
3,Athletic Club de Bilbao,Athletic Bilbao,San Mamés,Bilbao,43.26420,-2.94940
4,RC Deportivo de La Coruña,Deportivo La Coruña,Riazor,A Coruña,43.36860,-8.41890
5,RCD Espanyol,Espanyol,RCDE Stadium,Cornellí de Llobregat,41.34750,2.07560
6,Getafe CF,Getafe,Coliseum Alfonso Pérez,Getafe,40.32560,-3.71440
7,RC Recreativo,Recreativo,Estadio Nuevo Colombino,Huelva,37.24639,-6.95417
8,RCD Mallorca,Mallorca,Estadi Mallorca Son Moix,Palma,39.59000,2.63000
9,CD Numancia,Numancia,Nuevo Estadio Los Pajaritos,Soria,41.75444,-2.46778


### Parte 4

- Dependiendo del método para combinar los dataframes, el dataframe resultante podría tener registros duplicados, es decir, varios estadios relacionados con varios equipos. Dicho
esto, desarrolle una lógica para obtener un registro por equipo y estadio; este nuevo dataframe sin duplicados es nuestro nuevo dataframe **D**.

In [8]:
df_merge.columns

Index(['Equipo_longName', 'Club', 'Stadium', 'City', 'Latitude', 'Longitude'], dtype='object')

In [9]:
df_merge[df_merge.duplicated(subset='Equipo_longName', keep=False)]

,Equipo_longName,Club,Stadium,City,Latitude,Longitude
26,Real Valladolid,Real Madrid,Santiago Bernabéu,Madrid,40.4531,-3.6883
27,Real Valladolid,Valladolid,José Zorrilla,Valladolid,41.6458,-4.7611


In [10]:
df_d = df_merge.drop_duplicates(
    subset='Equipo_longName'
)

df_d

,Equipo_longName,Club,Stadium,City,Latitude,Longitude
0,UD Almería,Almería,Estadio de los Juegos Mediterráneos,Almería,36.84000,-2.46700
1,Atlético Madrid,Atlético Madrid,Metropolitano,Madrid,40.43620,-3.59950
2,FC Barcelona,Barcelona,Camp Nou,Barcelona,41.38090,2.12280
3,Athletic Club de Bilbao,Athletic Bilbao,San Mamés,Bilbao,43.26420,-2.94940
4,RC Deportivo de La Coruña,Deportivo La Coruña,Riazor,A Coruña,43.36860,-8.41890
5,RCD Espanyol,Espanyol,RCDE Stadium,Cornellí de Llobregat,41.34750,2.07560
6,Getafe CF,Getafe,Coliseum Alfonso Pérez,Getafe,40.32560,-3.71440
7,RC Recreativo,Recreativo,Estadio Nuevo Colombino,Huelva,37.24639,-6.95417
8,RCD Mallorca,Mallorca,Estadi Mallorca Son Moix,Palma,39.59000,2.63000
9,CD Numancia,Numancia,Nuevo Estadio Los Pajaritos,Soria,41.75444,-2.46778


In [11]:
equipos_faltantes = df_c[
    ~df_c['Equipo_longName'].isin(df_d['Equipo_longName'])
]

equipos_faltantes

,Equipo_longName
1230,Real Betis Balompié
1261,Xerez Club Deportivo
1270,Hércules Club de Fútbol


In [12]:
faltantes = df_b[
    df_b['Club'].isin([
        'Betis',
        'Xerez',
        'Hércules'
    ])
].copy()

faltantes

,Club,Stadium,City,Latitude,Longitude
6,Betis,Benito Villamarín,Seville,37.35640,-5.98160
35,Xerez,Estadio Municipal de Chapí­n,Jerez de la Frontera,36.68683,-6.11883
36,Hércules,Estadio José Rico Pérez,Alicante,38.35722,-0.49250


In [13]:
faltantes['Equipo_longName'] = [
    'Real Betis Balompié',
    'Xerez Club Deportivo',
    'Hércules Club de Fútbol'
]

faltantes

,Club,Stadium,City,Latitude,Longitude,Equipo_longName
6,Betis,Benito Villamarín,Seville,37.35640,-5.98160,Real Betis Balompié
35,Xerez,Estadio Municipal de Chapí­n,Jerez de la Frontera,36.68683,-6.11883,Xerez Club Deportivo
36,Hércules,Estadio José Rico Pérez,Alicante,38.35722,-0.49250,Hércules Club de Fútbol


In [14]:
df_d = pd.concat(
    [df_d, faltantes],
    ignore_index=True
)

df_d

,Equipo_longName,Club,Stadium,City,Latitude,Longitude
0,UD Almería,Almería,Estadio de los Juegos Mediterráneos,Almería,36.84000,-2.46700
1,Atlético Madrid,Atlético Madrid,Metropolitano,Madrid,40.43620,-3.59950
2,FC Barcelona,Barcelona,Camp Nou,Barcelona,41.38090,2.12280
3,Athletic Club de Bilbao,Athletic Bilbao,San Mamés,Bilbao,43.26420,-2.94940
4,RC Deportivo de La Coruña,Deportivo La Coruña,Riazor,A Coruña,43.36860,-8.41890
5,RCD Espanyol,Espanyol,RCDE Stadium,Cornellí de Llobregat,41.34750,2.07560
6,Getafe CF,Getafe,Coliseum Alfonso Pérez,Getafe,40.32560,-3.71440
7,RC Recreativo,Recreativo,Estadio Nuevo Colombino,Huelva,37.24639,-6.95417
8,RCD Mallorca,Mallorca,Estadi Mallorca Son Moix,Palma,39.59000,2.63000
9,CD Numancia,Numancia,Nuevo Estadio Los Pajaritos,Soria,41.75444,-2.46778


In [15]:
df_c[
    ~df_c['Equipo_longName'].isin(df_d['Equipo_longName'])
]

,Equipo_longName


### Parte 5

- Con la información ahora de los equipos y los estadios (D), combine ahora esos datos con el dataframe inicial (**A**) de la tabla de posiciones por temporada de la liga española (**A**), llamaremos a este dataframe final **F**.

In [16]:
df_f = spain_liga_bbva.merge(df_d, on='Equipo_longName', how='left')

In [17]:
df_f[df_f.isna().any(axis=1)]

,Liga,Temporada,Equipo,Equipo_longName,Total_Puntos,Partidos_Jugados,Partidos_Empatados,Partidos_Perdidos,GF,GC,Diferencia_Goles,Club,Stadium,City,Latitude,Longitude


In [18]:
df_f.isnull().sum()

Liga                  0
Temporada             0
Equipo                0
Equipo_longName       0
Total_Puntos          0
Partidos_Jugados      0
Partidos_Empatados    0
Partidos_Perdidos     0
GF                    0
GC                    0
Diferencia_Goles      0
Club                  0
Stadium               0
City                  0
Latitude              0
Longitude             0
dtype: int64

### Parte 6

- Seleccionando solo una de las temporadas (definida a su discreción), muestre el dataframe resultante **F**.

In [19]:
df_temp2015_2016 = df_f[
                         df_f['Temporada'] == '2015/2016'
                         ].reset_index(drop=True)  
df_temp2015_2016

,Liga,Temporada,Equipo,Equipo_longName,Total_Puntos,Partidos_Jugados,Partidos_Empatados,Partidos_Perdidos,GF,GC,Diferencia_Goles,Club,Stadium,City,Latitude,Longitude
0,Spain LIGA BBVA,2015/2016,AMA,Atlético Madrid,88,38,4,6,63,18,45,Atlético Madrid,Metropolitano,Madrid,40.4362,-3.5995
1,Spain LIGA BBVA,2015/2016,BAR,FC Barcelona,91,38,4,5,112,29,83,Barcelona,Camp Nou,Barcelona,41.3809,2.1228
2,Spain LIGA BBVA,2015/2016,BET,Real Betis Balompié,45,38,12,15,34,52,-18,Betis,Benito Villamarín,Seville,37.3564,-5.9816
3,Spain LIGA BBVA,2015/2016,BIL,Athletic Club de Bilbao,62,38,8,12,58,45,13,Athletic Bilbao,San Mamés,Bilbao,43.2642,-2.9494
4,Spain LIGA BBVA,2015/2016,CEL,RC Celta de Vigo,60,38,9,12,51,59,-8,Celta Vigo,Balaídos,Vigo,42.2118,-8.7394
5,Spain LIGA BBVA,2015/2016,COR,RC Deportivo de La Coruña,42,38,18,12,45,61,-16,Deportivo La Coruña,Riazor,A Coruña,43.3686,-8.4189
6,Spain LIGA BBVA,2015/2016,EIB,SD Eibar,43,38,10,17,49,61,-12,Eibar,Ipurua,Eibar,43.1833,-2.4667
7,Spain LIGA BBVA,2015/2016,ESP,RCD Espanyol,43,38,7,19,40,74,-34,Espanyol,RCDE Stadium,Cornellí de Llobregat,41.3475,2.0756
8,Spain LIGA BBVA,2015/2016,GET,Getafe CF,36,38,9,20,37,67,-30,Getafe,Coliseum Alfonso Pérez,Getafe,40.3256,-3.7144
9,Spain LIGA BBVA,2015/2016,GRA,Granada CF,39,38,9,19,46,69,-23,Granada,Nuevo Estadio de Los Cármenes,Granada,37.1496,-3.6020


<a id="ej2"></a>
# Ejercicio 2
[15 puntos] Utilizando `folium` y el dataframe **F**, muestre un mapa indicando todos los estadios de la Liga Española (`Spain LIGA BBVA`). En la etiqueta del marcador muestre el nombre del equipo y nombre del estadio.

[↑ Volver al Menú](#menu)

In [20]:
mapa = folium.Map(
    location=[40.4, -3],
    zoom_start=6,
    tiles = "cartodbPositron")

In [21]:
# icon_shadow = folium.Icon(
#     icon = "home",
#     shadow_size = (0, 0)
# )
for idx, row in df_f.iterrows():
    popup_text = (f"<strong>Estadio:</strong> <i>{row['Stadium']}</i>.<br>"
                  f"<strong>Equipo:</strong> <i>{row['Equipo_longName']}</i>.<br>")
    popup_config = folium.Popup(popup_text, max_width=400)
    
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=popup_config
        # icon = icon_shadow
    ).add_to(mapa)

mapa

<a id="ej3"></a>
# Ejercicio 3

[25 puntos] Utilizando `folium` y el dataframe **F**, muestre en un mapa la cantidad total de goles anotado como local por estadio (todas las temporadas), debe representar la cantidad de goles como el tamaño del marcador (`CircleMarker`) y el color del marcador debe estar en función al estadio. En la etiqueta del marcador muestre el nombre del estadio y el total de goles.

[↑ Volver al Menú](#menu)

In [22]:
df_goles_local = df_f.groupby(['Stadium', 'Latitude', 'Longitude'])['GF'].sum().reset_index()
df_goles_local.head()

,Stadium,Latitude,Longitude,GF
0,Balaídos,42.2118,-8.7394,184
1,Benito Villamarín,37.3564,-5.9816,225
2,Camp Nou,41.3809,2.1228,849
3,Campo de Fútbol de Vallecas,40.3919,-3.6590,247
4,Ciutat de Valíncia,39.4946,-0.3633,241


In [23]:
mapa_goles_local = folium.Map(
    location=[40.4, -3.7],
    zoom_start=6,
    tiles = "cartodbPositron")

for idx, row in df_goles_local.iterrows():
    
    popup_text=(f"<strong>Estadio:</strong> <i>{row['Stadium']}</i>.<br>"
                f"<strong>Total de Goles:</strong> <i>{row['GF']}</i>.<br>")
    popup_config=folium.Popup(popup_text, max_width=400)

    folium.CircleMarker(
        location=[
            row['Latitude'],
            row['Longitude']
        ],

        radius=row['GF'] / 30,
        # tooltip=row['Stadium'],
        popup=popup_config,
        
        
        fill=True,
        color='None', # borde
        fill_color='darkblue',
        fill_opacity=0.6

    ).add_to(mapa_goles_local)

mapa_goles_local

<a id="ej4"></a>
# Ejercicio 4

[35 puntos] Utilizando `folium` y el dataframe **F**, muestre en un mapa todos los estadios, similar al ejercicio #2, pero esta vez al hacer clic en el marcador se debe mostrar un subgráfico con `seaborn` mostrando un gráfico de barras mostrando con el total de goles a favor y en contra para la temporada ‘2014/2015’.

[↑ Volver al Menú](#menu)

In [24]:
df_temp2014_2015 = df_f[
                         df_f['Temporada'] == '2014/2015'
                         ].reset_index(drop=True)  
df_temp2014_2015.sample(5)

,Liga,Temporada,Equipo,Equipo_longName,Total_Puntos,Partidos_Jugados,Partidos_Empatados,Partidos_Perdidos,GF,GC,Diferencia_Goles,Club,Stadium,City,Latitude,Longitude
16,Spain LIGA BBVA,2014/2015,SOC,Real Sociedad,46,38,13,14,44,51,-7,Real Sociedad,Reale Arena (Anoeta),San Sebastián,43.3018,-1.9736
7,Spain LIGA BBVA,2014/2015,ELC,Elche CF,41,38,8,19,35,62,-27,Elche,Estadio Manuel Martínez Valero,Elche,38.2699,-0.7126
13,Spain LIGA BBVA,2014/2015,RAY,Rayo Vallecano,49,38,4,19,46,68,-22,Rayo Vallecano,Campo de Fútbol de Vallecas,Madrid,40.3919,-3.6590
3,Spain LIGA BBVA,2014/2015,BIL,Athletic Club de Bilbao,55,38,10,13,42,41,1,Athletic Bilbao,San Mamés,Bilbao,43.2642,-2.9494
9,Spain LIGA BBVA,2014/2015,GET,Getafe CF,37,38,7,21,33,64,-31,Getafe,Coliseum Alfonso Pérez,Getafe,40.3256,-3.7144


In [25]:
df_14_15 = df_temp2014_2015.groupby(['Stadium', 'Latitude', 'Longitude'])[['GF', 'GC']].sum().reset_index()
df_14_15.head()

,Stadium,Latitude,Longitude,GF,GC
0,Balaídos,42.2118,-8.7394,47,44
1,Camp Nou,41.3809,2.1228,110,21
2,Campo de Fútbol de Vallecas,40.3919,-3.6590,46,68
3,Ciutat de Valíncia,39.4946,-0.3633,34,67
4,Coliseum Alfonso Pérez,40.3256,-3.7144,33,64


In [26]:
def grafico_barras(row):
    fig, ax = plt.subplots(figsize=(3,3))
    categorias = ['GF', 'GC']
    valores = [row['GF'], row['GC']]

    sns.barplot(
        x=categorias,
        y=valores,
        ax=ax
    )
    sns.despine(left=True, bottom=False) # Deja visible solo el eje X

    ax.set_title(f"Estadio\n{row['Stadium']}")
    ax.bar_label(ax.containers[0], padding=1, fontsize=11, label_type='center', color='white')
    ax.get_yaxis().set_visible(False) 
    # ax.set_xlabel("Total Goles")
    # ax.set_ylabel("Cantidad")
    img = BytesIO()

    plt.savefig(
        img,
        format='png',
        bbox_inches='tight'
    )

    # plt.tight_layout()
    # plt.show
    plt.close()
    
    # Convertir imagen a base64
    img_base64 = base64.b64encode(img.getvalue()).decode()
    html = f'''<img src="data:image/png;base64,{img_base64}">'''

    return html

In [27]:
# for idx, row in df_14_15.iterrows():
#     test = row

# grafico_barras(test)

In [28]:
mapa_stats = folium.Map(
    location=[40.4, -3.7],
    zoom_start=6,
    tiles = "cartodbPositron")

for idx, row in df_14_15.iterrows():

    # Crear html del gráfico
    html = grafico_barras(row)

    # Popup
    iframe = folium.IFrame(
        html=html,
        width=450,
        height=350)

    popup_config = folium.Popup(
        iframe,
        max_width=300)

    # Marker
    folium.Marker(
        location=[
            row['Latitude'],
            row['Longitude']
        ], popup=popup_config,

        tooltip=f"<strong>Estadio:</strong> <i>{row['Stadium']}</i>"

    ).add_to(mapa_stats)
    
mapa_stats